In [71]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import sqlite3

# API for building permit dataset.
url = (
    "https://opendata.vancouver.ca/api/explore/v2.1/"
    "catalog/datasets/issued-building-permits/exports/csv"
)

response = requests.get(url)
response.raise_for_status()

# Turn API data into Pandas dataframe.
df = pd.read_csv(StringIO(response.text), sep=";")

print(df.shape)
print(df.columns.tolist())

(51848, 20)
['permitnumber', 'permitnumbercreateddate', 'issuedate', 'permitelapseddays', 'projectvalue', 'typeofwork', 'address', 'projectdescription', 'permitcategory', 'applicant', 'applicantaddress', 'propertyuse', 'specificusecategory', 'buildingcontractor', 'buildingcontractoraddress', 'issueyear', 'geolocalarea', 'geom', 'yearmonth', 'geo_point_2d']


In [72]:
df = df.rename(columns={
    "permitnumber": "permit_number",
    "permitnumbercreateddate": "permit_number_created_date",
    "issuedate": "issue_date",
    "permitelapseddays": "permit_elapsed_days",
    "projectvalue": "project_value",
    "typeofwork": "type_of_work",
    "address": "address",
    "projectdescription": "project_description",
    "permitcategory": "permit_category",
    "applicant": "applicant",
    "applicantaddress": "applicant_address",
    "propertyuse": "property_use",
    "specificusecategory": "specific_use_category",
    "buildingcontractor": "building_contractor",
    "buildingcontractoraddress": "building_contractor_address",
    "issueyear": "issue_year",
    "geolocalarea": "geo_local_area",
    "geom": "geom",
    "yearmonth": "year_month",
    "geo_point_2d": "geo_point_2d"
})

In [73]:
print(df.columns.tolist())
print(df.shape)

['permit_number', 'permit_number_created_date', 'issue_date', 'permit_elapsed_days', 'project_value', 'type_of_work', 'address', 'project_description', 'permit_category', 'applicant', 'applicant_address', 'property_use', 'specific_use_category', 'building_contractor', 'building_contractor_address', 'issue_year', 'geo_local_area', 'geom', 'year_month', 'geo_point_2d']
(51848, 20)


In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51848 entries, 0 to 51847
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   permit_number                51848 non-null  str    
 1   permit_number_created_date   51848 non-null  str    
 2   issue_date                   51848 non-null  str    
 3   permit_elapsed_days          51848 non-null  int64  
 4   project_value                51848 non-null  float64
 5   type_of_work                 51848 non-null  str    
 6   address                      51635 non-null  str    
 7   project_description          51848 non-null  str    
 8   permit_category              29732 non-null  str    
 9   applicant                    51848 non-null  str    
 10  applicant_address            51673 non-null  str    
 11  property_use                 51843 non-null  str    
 12  specific_use_category        51841 non-null  str    
 13  building_contractor        

In [75]:
# Convert numeric fields.
df['permit_elapsed_days'] = pd.to_numeric(
    df['permit_elapsed_days'],
    errors='coerce'
)

df['project_value'] = pd.to_numeric(
    df['project_value'],
    errors='coerce'
)

df['issue_year'] = pd.to_numeric(
    df['issue_year'],
    errors='coerce'
)

print(df[['permit_elapsed_days', 'project_value', 'issue_year']].dtypes)

permit_elapsed_days      int64
project_value          float64
issue_year               int64
dtype: object


In [76]:
# Remove records without coordinates.
df = df.dropna(subset=['geo_point_2d'])

# Split coordinates into latitude and longitude.
df[['latitude', 'longitude']] = (
    df['geo_point_2d']
    .str.split(',', expand=True)
)

df['latitude'] = pd.to_numeric(
    df['latitude'].str.strip(),
    errors='coerce'
)

df['longitude'] = pd.to_numeric(
    df['longitude'].str.strip(),
    errors='coerce'
)

# Remove invalid coordinates.
df = df.dropna(subset=['latitude', 'longitude'])

df = df[
    df['latitude'].between(49.0, 49.5) &
    df['longitude'].between(-123.5, -122.5)
].copy()

# The GeoJSON-style column is no longer necessary.
df = df.drop(columns=['geo_point_2d'])

print(df[['latitude', 'longitude']].head())

    latitude   longitude
0  49.222269 -123.072869
1  49.226715 -123.031846
2  49.285344 -123.129665
3  49.215506 -123.068972
4  49.226026 -123.071232


In [77]:
# Final data check.

print("Final dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate permit numbers:")
print(df['permit_number'].duplicated().sum())

print("\nProject value summary:")
print(df['project_value'].describe())

print("\nPermit elapsed days summary:")
print(df['permit_elapsed_days'].describe())

print("\nSample:")
display(df.head())

Final dataset shape: (51488, 21)

Data types:
permit_number                      str
permit_number_created_date         str
issue_date                         str
permit_elapsed_days              int64
project_value                  float64
type_of_work                       str
address                            str
project_description                str
permit_category                    str
applicant                          str
applicant_address                  str
property_use                       str
specific_use_category              str
building_contractor                str
building_contractor_address        str
issue_year                       int64
geo_local_area                     str
geom                               str
year_month                         str
latitude                       float64
longitude                      float64
dtype: object

Missing values:
permit_number                      0
permit_number_created_date         0
issue_date                    

,permit_number,permit_number_created_date,issue_date,permit_elapsed_days,project_value,type_of_work,address,project_description,permit_category,applicant,...,property_use,specific_use_category,building_contractor,building_contractor_address,issue_year,geo_local_area,geom,year_month,latitude,longitude
0,BP-2019-04562,2019-10-09,2019-10-28,19,0.0,Salvage and Abatement,"6836 FLEMING STREET, Vancouver, BC V5P 3H5",Low Density Housing - Salvage and Abatement - ...,NaN,Mukhtiar Sian DBA: Sian Group Investments Inc.,...,Dwelling Uses,Single Detached House,NaN,NaN,2019,Victoria-Fraserview,"{""coordinates"": [-123.0728689, 49.2222693], ""t...",2019-10,49.222269,-123.072869
1,DB-2019-03291,2019-07-24,2020-02-05,196,15000.0,Demolition / Deconstruction,"3407 E 47TH AVENUE, Vancouver, BC V5S 1C8",Low Density Housing - Demolition / Deconstruct...,NaN,Mukhtiar Sian DBA: Sian Group Investments Inc.,...,Dwelling Uses,Single Detached House,G & K Excavation and Demolition Services Ltd,NaN,2020,Killarney,"{""coordinates"": [-123.0318457, 49.2267148], ""t...",2020-02,49.226715,-123.031846
2,BP-2018-06445,2018-12-12,2019-01-24,43,8000.0,Addition / Alteration,"1263 BARCLAY STREET #406, Vancouver, BC V6E 1H5",Field Review - Addition / Alteration - #406\r\...,Renovation - Residential - Lower Complexity,daniel Charette,...,Dwelling Uses,Multiple Dwelling,JFL Renovation Innovation Inc,"2980 E 6TH AV \r\nVancouver, BC V5M 1S1",2019,West End,"{""coordinates"": [-123.1296648, 49.2853443], ""t...",2019-01,49.285344,-123.129665
3,DB-2025-02049,2025-05-01,2025-07-15,75,248000.0,Addition / Alteration,"7611 THORNHILL DRIVE, Vancouver, BC V5P 3T3",Low Density Housing - Addition / Alteration - ...,NaN,Mukhtiar Sian DBA: Sian Group Investments Inc.,...,Dwelling Uses,Laneway House,NaN,NaN,2025,Victoria-Fraserview,"{""coordinates"": [-123.0689721, 49.2155063], ""t...",2025-07,49.215506,-123.068972
4,DB-2017-05083,2017-09-27,2018-04-18,203,1106000.0,New Building,"6434 ARGYLE STREET, Vancouver, BC V5P 3K3",Low Density Housing - New Building - To constr...,New Build - Low Density Housing,Mukhtiar Sian DBA: Sian Group Investments Inc.,...,Dwelling Uses,Single Detached House w/Sec Suite,Sian Group Investments Inc,"2177 BONACCORD DRIVE \r\nVancouver, BC V5P 2N8",2018,Victoria-Fraserview,"{""coordinates"": [-123.0712319, 49.2260263], ""t...",2018-04,49.226026,-123.071232


In [35]:
# Connect to SQLite database
conn = sqlite3.connect('building_permits.db')

# Write cleaned DataFrame to SQLite
df.to_sql(
    'permits',
    conn,
    if_exists='replace',
    index=False
)

print("Data successfully saved to SQLite.")

Data successfully saved to SQLite.


In [ ]:
# QUERY 1: Where is construction activity concentrated?

query_1 = """
SELECT 
    geo_local_area,
    COUNT(*) AS total_permits,
    SUM(project_value) AS total_project_value,
    AVG(project_value) AS average_project_value
FROM permits
WHERE project_value > 0 
    AND project_value IS NOT NULL
    AND geo_local_area IS NOT NULL
GROUP BY geo_local_area
ORDER BY total_permits DESC;
"""

result = pd.read_sql_query(query_1, conn)

display(result)

,geo_local_area,total_permits,total_project_value,average_project_value
0,Downtown,7384,7.295040e+09,9.879523e+05
1,Kensington-Cedar Cottage,3150,1.710775e+09,5.431030e+05
2,Renfrew-Collingwood,2590,2.014745e+09,7.778940e+05
3,Hastings-Sunrise,2527,1.328658e+09,5.257845e+05
4,West End,2390,2.307352e+09,9.654191e+05
5,Kitsilano,2327,1.536374e+09,6.602381e+05
6,Sunset,2289,9.502790e+08,4.151503e+05
7,Fairview,2225,1.768528e+09,7.948441e+05
8,Dunbar-Southlands,2013,1.134198e+09,5.634364e+05
9,Riley Park,1928,1.621968e+09,8.412696e+05


In [53]:
# QUERY 2: Which specific property-use categories have the longest
# average processing times?

query_2 = """
WITH category_stats AS (
    SELECT
        specific_use_category,
        COUNT(*) AS total_permits,
        AVG(permit_elapsed_days) AS average_processing_days
    FROM permits
    WHERE permit_elapsed_days >= 0
        AND specific_use_category IS NOT NULL
    GROUP BY specific_use_category
)

SELECT 
    specific_use_category,
    total_permits,
    average_processing_days
FROM category_stats
WHERE total_permits >= 20
ORDER by average_processing_days DESC;
"""

result = pd.read_sql_query(query_2, conn)

display(result)

,specific_use_category,total_permits,average_processing_days
0,"Multiple Dwelling,Parking Garage",58,552.120690
1,"Secondary Suite,Duplex w/Secondary Suite",47,285.446809
2,Infill,40,284.175000
3,"Duplex w/Secondary Suite,Secondary Suite",123,272.918699
4,"Retail Store,Multiple Dwelling",21,266.761905
...,...,...,...
80,Health Care Office,633,55.263823
81,Farmers Market,33,51.303030
82,General Office,5462,50.859575
83,Museum or Archives,62,31.887097


In [11]:
# QUERY 3: Do larger projects take longer to process?

query_3 = """
SELECT
    CASE
        WHEN project_value < 10000 THEN 'Small'
        WHEN project_value < 100000 THEN 'Medium'
        ELSE 'Large'
    END AS project_size,
    COUNT(*) AS total_permits,
    AVG(permit_elapsed_days) AS average_processing_days
FROM permits
WHERE project_value > 0
    AND permit_elapsed_days >= 0
GROUP BY project_size
ORDER BY average_processing_days DESC;
"""

result = pd.read_sql_query(query_3, conn)

display(result)

,project_size,total_permits,average_processing_days
0,Large,21368,173.123222
1,Medium,18827,115.709194
2,Small,3395,65.756701


In [55]:
# QUERY 4: How has construction activity changed over time?

query_4 = """
SELECT
    issue_year,
    COUNT(*) AS total_permits,
    SUM(project_value) AS total_project_value
FROM permits
WHERE project_value > 0
    AND issue_year IS NOT NULL
GROUP BY issue_year
ORDER BY issue_year DESC;
"""

result = pd.read_sql_query(query_4, conn)

display(result)

ProgrammingError: Cannot operate on a closed database.

In [ ]:
# QUERY 5: Are certain geographic areas associated with longer permit
# processing times?

query_5 = """
SELECT
    geo_local_area,
    AVG(permit_elapsed_days) AS average_processing_time,
    COUNT(*) AS total_permits
FROM permits
WHERE permit_elapsed_days >= 0
    AND geo_local_area IS NOT NULL
GROUP BY geo_local_area
HAVING COUNT(*) >= 20
ORDER BY average_processing_time DESC;
"""

result = pd.read_sql_query(query_5, conn)

display(result)

,geo_local_area,average_processing_time,total_permits
0,South Cambie,178.728042,945
1,Oakridge,177.989362,1316
2,West Point Grey,168.688959,1585
3,Kerrisdale,168.247281,1379
4,Killarney,163.452143,1703
5,Renfrew-Collingwood,162.569231,3315
6,Sunset,162.360732,2786
7,Victoria-Fraserview,161.801831,2185
8,Dunbar-Southlands,160.756569,2588
9,Marpole,159.822303,1919


In [ ]:
# QUERY 6: Which geographic areas had the highest construction 
# investment each year?

query_6 = """
WITH area_investment AS (
    SELECT
        issue_year,
        geo_local_area,
        SUM(project_value) AS total_investment
    FROM permits
    WHERE project_value > 0
        AND geo_local_area IS NOT NULL
    GROUP BY issue_year, geo_local_area
),

ranked_areas AS (
    SELECT 
        issue_year,
        geo_local_area,
        total_investment,
        RANK () OVER (
            PARTITION BY issue_year
            ORDER BY total_investment DESC
        ) AS investment_rank
    FROM area_investment
)

SELECT 
    issue_year,
    geo_local_area,
    total_investment
FROM ranked_areas
WHERE investment_rank = 1
ORDER BY issue_year DESC;
"""

result = pd.read_sql_query(query_6, conn)

display(result)

,issue_year,geo_local_area,total_investment
0,2026,Kitsilano,6.381121e+08
1,2025,Downtown,1.037327e+09
2,2024,Downtown,5.269131e+08
3,2023,Downtown,1.253290e+09
4,2022,Strathcona,1.733265e+09
5,2021,Oakridge,7.789906e+08
6,2020,Downtown,3.485083e+08
7,2019,Downtown,1.394305e+09
8,2018,Downtown,8.321659e+08
9,2017,Downtown,4.574998e+08


In [54]:
conn.close()

In [78]:
# Create a cleaner dataset for Tableau.

tableau_df = df[
  [
    'permit_number',
    'permit_number_created_date',
    'issue_date',
    'permit_elapsed_days',
    'project_value',
    'type_of_work',
    'specific_use_category',
    'property_use',
    'issue_year',
    'geo_local_area',
    'year_month',
    'latitude',
    'longitude'
    ]
].copy()

tableau_df.to_csv(
  "building_permits_tableau.csv",
  index=False
)